코드 12
ViT(Vision Transformer) 단독 성능 재측정(잘못 측정된 성능 수정)
+ EfficientNet ensemble



In [1]:
import os
os.environ['OPENCV_LOG_LEVEL'] = 'ERROR' 

import json
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import random
import math

# ================= 설정 =================
RAW_IMG_BASE = "/workspace/user1/dataset_v2/img/train" 
LBL_BASE = "/workspace/user1/dataset_v2/label_mod/train"
VAL_IMG_BASE = "/workspace/user1/dataset_v2/img/val"
VAL_LBL_BASE = "/workspace/user1/dataset_v2/label_mod/val"
AUG_IMG_BASE = "/workspace/user5/img_augmented/train"

SAVE_DIR = '/workspace/user1/model_v8_arcface'
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
BATCH_SIZE = 64
VIT_IMG_SIZE = 224 # ViT 입력 규격
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

STAGE2_MAP = {'anger': 0, 'happy': 1, 'panic': 2, 'sadness': 3}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print(f"✅ 환경 설정 완료. 장치: {DEVICE}")

✅ 환경 설정 완료. 장치: cuda


In [2]:
# 1. ViT 전용 전처리
vit_train_transform = A.Compose([
    A.Resize(VIT_IMG_SIZE, VIT_IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.CoarseDropout(max_holes=2, max_height=32, max_width=32, fill_value=0, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

vit_val_transform = A.Compose([
    A.Resize(VIT_IMG_SIZE, VIT_IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 2. 크롭 로직이 포함된 원본 데이터셋 클래스
class ArcFaceStage2Dataset(Dataset):
    def __init__(self, raw_img_base, aug_img_base, lbl_base, transform=None, is_train=True):
        self.transform = transform
        self.samples = []
        counts = {0: 0, 1: 0, 2: 0, 3: 0}

        if os.path.exists(lbl_base):
            for json_file in tqdm(os.listdir(lbl_base), desc="Loading JSONs"):
                if not json_file.endswith('.json'): continue
                with open(os.path.join(lbl_base, json_file), 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    for item in data['images']:
                        if item.get('is_box_issue') == 'Yes': continue
                        emotion = item['final_emotion']
                        if emotion in ['neutral', 'blank']: continue
                        
                        label = STAGE2_MAP[emotion]
                        counts[label] += 1
                        self.samples.append({
                            'source': 'raw',
                            'img_path': os.path.join(raw_img_base, emotion, item['filename']),
                            'bbox': item['bbox'],
                            'label': label
                        })

        if is_train and os.path.exists(aug_img_base):
            for emotion_name in STAGE2_MAP.keys():
                src_folder = os.path.join(aug_img_base, emotion_name)
                if not os.path.isdir(src_folder): continue
                
                label = STAGE2_MAP[emotion_name]
                for filename in os.listdir(src_folder):
                    counts[label] += 1
                    self.samples.append({
                        'source': 'aug',
                        'img_path': os.path.join(src_folder, filename),
                        'label': label
                    })

        print(f"✅ 로딩 완료. {counts}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = cv2.imread(item['img_path'])
        if image is None: return self.__getitem__(random.randint(0, len(self)-1))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # JSON Box 좌표 기반 Crop 적용
        if item['source'] == 'raw':
            box = item['bbox']
            x1, y1, x2, y2 = max(0, box['minX']), max(0, box['minY']), box['maxX'], box['maxY']
            image = image[y1:y2, x1:x2]
            
        if self.transform:
            image = self.transform(image=image)['image']
            
        return image, torch.tensor(item['label'], dtype=torch.long)

# 3. 로더 생성
train_dataset = ArcFaceStage2Dataset(RAW_IMG_BASE, AUG_IMG_BASE, LBL_BASE, transform=vit_train_transform, is_train=True)
val_dataset = ArcFaceStage2Dataset(VAL_IMG_BASE, AUG_IMG_BASE, VAL_LBL_BASE, transform=vit_val_transform, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

/home/user1/.conda/envs/mwkim/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_3106994/1406940902.py:7: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=2, max_height=32, max_width=32, fill_value=0, p=0.5),
Loading JSONs: 100%|██████████| 5/5 [00:02<00:00,  1.94it/s]


✅ 로딩 완료. {0: 4627, 1: 7189, 2: 7706, 3: 8003}


Loading JSONs: 100%|██████████| 5/5 [00:00<00:00, 53.65it/s]

✅ 로딩 완료. {0: 184, 1: 281, 2: 303, 3: 269}


In [3]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50, easy_margin=False):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.easy_margin = easy_margin
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, label):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
        one_hot = torch.zeros(cosine.size(), device=DEVICE)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        
        return output, cosine

class ViTArcFace(nn.Module):
    def __init__(self, num_classes=4, m=0.5):
        super(ViTArcFace, self).__init__()
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        embed_dim = self.backbone.num_features 
        
        # 이후 앙상블 코드와 통일하기 위해 변수명을 arcface로 지정합니다.
        self.arcface = ArcMarginProduct(in_features=embed_dim, out_features=num_classes, s=30.0, m=m)

    def forward(self, x, labels=None):
        features = self.backbone(x)
        if labels is not None:
            output, cosine = self.arcface(features, labels)
            return output, cosine
        
        cosine = F.linear(F.normalize(features), F.normalize(self.arcface.weight))
        return cosine * self.arcface.s

In [4]:
model = ViTArcFace(num_classes=4, m=0.5).to(DEVICE)

# 블록 0~5번 동결, 6~11번 학습 허용
for name, param in model.backbone.named_parameters():
    if 'patch_embed' in name or 'cls_token' in name or 'pos_embed' in name:
        param.requires_grad = False
    if 'blocks' in name:
        block_idx = int(name.split('.')[1])
        if block_idx < 6: 
            param.requires_grad = False

trainable_params = filter(lambda p: p.requires_grad, model.parameters())

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(trainable_params, lr=3e-5, weight_decay=5e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_3106994/1149925241.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [5]:
epochs = 50
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs, cosine = model(images, labels)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        _, predicted = cosine.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    train_acc = correct / total
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            with torch.cuda.amp.autocast():
                outputs, cosine = model(images, labels)
                loss = criterion(outputs, labels)
                
            val_loss += loss.item()
            _, predicted = cosine.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    val_acc = correct / total
    scheduler.step()
    
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {train_loss/len(train_loader):.4f} Train Acc: {train_acc:.4f} | Val Loss: {val_loss/len(val_loader):.4f} Val Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # 저장 파일명을 명확히 지정합니다.
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'vit_arcface_sync_best.pth'))
        print(f"★ Best Model Saved! (Acc: {val_acc:.4f})")

Epoch 1/50 [Train]:   0%|          | 0/431 [00:00<?, ?it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
/tmp/ipykernel_3106994/731080566.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/50 [Train]:   3%|▎         | 15/431 [01:13<19:06,  2.76s/it] Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 1/50 [Train]:   9%|▉         | 39/431 [02:13<03:58,  1.64it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 1/50 [Train]:  15%|█▍        | 63/431 [02:28<02:18,  2.67it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 1/50 [Train]:  15

Epoch [1/50] Train Loss: 6.1628 Train Acc: 0.7647 | Val Loss: 4.3517 Val Acc: 0.8293
★ Best Model Saved! (Acc: 0.8293)


Epoch 2/50 [Train]:   2%|▏         | 8/431 [00:06<02:11,  3.22it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 2/50 [Train]:   4%|▎         | 16/431 [00:11<02:23,  2.89it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 2/50 [Train]:   6%|▌         | 24/431 [00:16<01:43,  3.93it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 2/50 [Train]:   7%|▋         | 31/431 [00:21<02:36,  2.55it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 2/50 [Train]:   9%|▉         | 40/431 [00:26<01:48,  3.62it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 2/50 [Train]:  14%|█▎        | 59/431 [00:39<03:26,  1.80it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 2/50 [Train]:  16%|█▌       

Epoch [2/50] Train Loss: 3.7077 Train Acc: 0.8450 | Val Loss: 3.8472 Val Acc: 0.8486
★ Best Model Saved! (Acc: 0.8486)


Epoch 3/50 [Train]:   2%|▏         | 7/431 [00:06<02:51,  2.47it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 3/50 [Train]:   3%|▎         | 15/431 [00:11<02:54,  2.39it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 3/50 [Train]:   4%|▍         | 17/431 [00:15<06:07,  1.13it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 3/50 [Train]:   5%|▌         | 23/431 [00:16<01:58,  3.45it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 3/50 [Train]:   7%|▋         | 31/431 [00:22<02:25,  2.75it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 3/50 [Train]:  11%|█         | 48/431 [00:31<01:50,  

Epoch [3/50] Train Loss: 2.9393 Train Acc: 0.8728 | Val Loss: 3.6758 Val Acc: 0.8486


... (90 outputs left out)

Epoch 49/50 [Train]:   0%|          | 0/431 [00:00<?, ?it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 49/50 [Train]:   4%|▎         | 16/431 [00:11<01:59,  3.47it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 49/50 [Train]:  20%|██        | 88/431 [00:54<02:08,  2.67it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 49/50 [Train]:  22%|██▏       | 96/431 [00:59<01:47,  3.13it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 49/50 [Train]:  26%|██▌       | 110/431 [01:07<01:44,  3.08it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 49/50 [Train]:  30%|██▉       | 128/431 [01:18<02:04,  2.43it/s]Invalid SOS parameters for sequent

Epoch [49/50] Train Loss: 2.5981 Train Acc: 0.8666 | Val Loss: 3.6777 Val Acc: 0.8149


Epoch 50/50 [Train]:   5%|▌         | 23/431 [00:15<02:09,  3.14it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 50/50 [Train]:   7%|▋         | 32/431 [00:20<01:39,  4.00it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 50/50 [Train]:  10%|▉         | 41/431 [00:28<05:00,  1.30it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 50/50 [Train]:  11%|█         | 47/431 [00:30<02:28,  2.58it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 50/50 [Train]:  19%|█▉        | 82/431 [00:52<03:57,  1.47it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 50/50 [Train]:  22%|██▏       | 96/431 [00:59<02:02,  2.74it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 50/50 [Train]:  25%|█

Epoch [50/50] Train Loss: 2.6025 Train Acc: 0.8679 | Val Loss: 3.7014 Val Acc: 0.8149


In [10]:
import torch.nn.functional as F

# 1. EfficientNet + ArcFace 클래스 선언 (과거 변수명 arcface 맞춤 버전)
class EffNetArcFace(nn.Module):
    def __init__(self, num_classes=4):
        super(EffNetArcFace, self).__init__()
        self.backbone = timm.create_model('efficientnet_b2', pretrained=False, num_classes=0)
        self.arcface = ArcMarginProduct(in_features=1408, out_features=num_classes, s=30.0, m=0.7)

    def forward(self, x):
        features = self.backbone(x)
        # 추론 시에는 페널티 없이 순수 유사도 반환
        cosine = F.linear(F.normalize(features), F.normalize(self.arcface.weight))
        return cosine * self.arcface.s

# 2. 앙상블 모델 가중치 로드
print("모델 가중치 로드 중...")
eff_model = EffNetArcFace(num_classes=4).to(DEVICE)
eff_model.load_state_dict(torch.load('/workspace/user1/model_v8_arcface/best_stage2_arcface.pth', map_location=DEVICE))
eff_model.eval()

vit_model = ViTArcFace(num_classes=4, m=0.5).to(DEVICE)
vit_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, '/workspace/user1/model_v8_arcface/vit_arcface_sync_best.pth'), map_location=DEVICE))
vit_model.eval()

# 3. 앙상블 전용 데이터 로더 준비 (기준 사이즈: 260)
ens_val_transform = A.Compose([
    A.Resize(260, 260),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

ens_val_dataset = ArcFaceStage2Dataset(VAL_IMG_BASE, AUG_IMG_BASE, VAL_LBL_BASE, transform=ens_val_transform, is_train=False)
ens_val_loader = DataLoader(ens_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

# 4. 앙상블 추론 (가중치 0.6 : 0.4)
WEIGHT_EFF = 0.6
WEIGHT_VIT = 0.4

correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(ens_val_loader, desc="[Ensemble Inference]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        with torch.amp.autocast('cuda'):
            # EfficientNet 추론 (260x260)
            eff_cosine = eff_model(images)
            
            # ViT 추론 (224x224 리사이징)
            vit_images = F.interpolate(images, size=(224, 224), mode='bilinear', align_corners=False)
            vit_cosine = vit_model(vit_images, labels=None)
            
            # 확률 변환 및 Soft Voting
            prob_eff = F.softmax(eff_cosine, dim=1)
            prob_vit = F.softmax(vit_cosine, dim=1)
            
            ensemble_prob = (prob_eff * WEIGHT_EFF) + (prob_vit * WEIGHT_VIT)
            
            _, predicted = ensemble_prob.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

print(f"\n=== 최종 앙상블 결과 ===")
print(f"Ensemble Validation Acc : {correct / total:.4f}")

모델 가중치 로드 중...


Loading JSONs: 100%|██████████| 5/5 [00:00<00:00, 75.31it/s]


✅ 로딩 완료. {0: 184, 1: 281, 2: 303, 3: 269}


[Ensemble Inference]:   0%|          | 0/17 [00:00<?, ?it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential


=== 최종 앙상블 결과 ===
Ensemble Validation Acc : 0.8698


In [11]:
import torch.nn.functional as F
import torch

WEIGHT_EFF = 0.6
WEIGHT_VIT = 0.4

all_eff_logits = []
all_vit_logits = []
all_labels = []

# 1. 모델들의 순수 출력값(Logits) 수집
with torch.no_grad():
    for images, labels in tqdm(ens_val_loader, desc="[Extracting Logits]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        with torch.amp.autocast('cuda'):
            eff_cosine = eff_model(images)
            vit_images = F.interpolate(images, size=(224, 224), mode='bilinear', align_corners=False)
            vit_cosine = vit_model(vit_images, labels=None)
            
            # Softmax 없이 순수 값을 CPU로 내려서 저장
            all_eff_logits.append(eff_cosine.cpu())
            all_vit_logits.append(vit_cosine.cpu())
            all_labels.append(labels.cpu())

# 리스트를 하나의 텐서로 병합
all_eff_logits = torch.cat(all_eff_logits)
all_vit_logits = torch.cat(all_vit_logits)
all_labels = torch.cat(all_labels)

# --- 단일 모델 성능 확인 ---
_, eff_preds = all_eff_logits.max(1)
eff_acc = eff_preds.eq(all_labels).sum().item() / len(all_labels)

_, vit_preds = all_vit_logits.max(1)
vit_acc = vit_preds.eq(all_labels).sum().item() / len(all_labels)

print("\n=== 단일 모델 성능 ===")
print(f"EfficientNet 단독 : {eff_acc:.4f}")
print(f"ViT 단독          : {vit_acc:.4f}")

# 2. 가중치 전수 조사 (Grid Search)
print("\n=== 앙상블 가중치 전수 조사 (Softmax 제거) ===")
best_acc = 0.0
best_weight = 0.0

# EfficientNet 가중치를 0.1부터 0.9까지 테스트
for w in range(1, 10):
    w_eff = w / 10.0
    w_vit = 1.0 - w_eff
    
    # 🔴 Softmax 없이 각도(유사도)를 바로 합산
    ensemble_logits = (all_eff_logits * w_eff) + (all_vit_logits * w_vit)
    _, ens_preds = ensemble_logits.max(1)
    
    acc = ens_preds.eq(all_labels).sum().item() / len(all_labels)
    print(f"Eff {w_eff:.1f} : ViT {w_vit:.1f} -> Acc: {acc:.4f}")
    
    if acc > best_acc:
        best_acc = acc
        best_weight = w_eff

print(f"\n★ 최고 앙상블 성능: {best_acc:.4f} (Eff 가중치: {best_weight:.1f})")

[Extracting Logits]:   0%|          | 0/17 [00:00<?, ?it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
[Extracting Logits]:  18%|█▊        | 3/17 [00:05<00:21,  1.52s/it]Invalid SOS pa


=== 단일 모델 성능 ===
EfficientNet 단독 : 0.8621
ViT 단독          : 0.8563

=== 앙상블 가중치 전수 조사 (Softmax 제거) ===
Eff 0.1 : ViT 0.9 -> Acc: 0.8621
Eff 0.2 : ViT 0.8 -> Acc: 0.8602
Eff 0.3 : ViT 0.7 -> Acc: 0.8602
Eff 0.4 : ViT 0.6 -> Acc: 0.8650
Eff 0.5 : ViT 0.5 -> Acc: 0.8708
Eff 0.6 : ViT 0.4 -> Acc: 0.8708
Eff 0.7 : ViT 0.3 -> Acc: 0.8650
Eff 0.8 : ViT 0.2 -> Acc: 0.8640
Eff 0.9 : ViT 0.1 -> Acc: 0.8660

★ 최고 앙상블 성능: 0.8708 (Eff 가중치: 0.5)


In [12]:
import torch
import torch.nn.functional as F

all_eff_tta_logits = []
all_vit_tta_logits = []
all_labels = []

# 1. TTA를 적용한 Logit 추출
with torch.no_grad():
    for images, labels in tqdm(ens_val_loader, desc="[Extracting TTA Logits]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        # 좌우 반전 이미지 생성
        images_flipped = torch.flip(images, dims=[3])
        
        with torch.amp.autocast('cuda'):
            # --- (A) EfficientNet TTA 추론 ---
            eff_out_orig = eff_model(images)
            eff_out_flip = eff_model(images_flipped)
            # 두 출력값의 평균을 내어 TTA 효과 적용
            eff_tta_cosine = (eff_out_orig + eff_out_flip) / 2.0
            
            # --- (B) ViT TTA 추론 ---
            vit_images_orig = F.interpolate(images, size=(224, 224), mode='bilinear', align_corners=False)
            vit_images_flip = F.interpolate(images_flipped, size=(224, 224), mode='bilinear', align_corners=False)
            
            vit_out_orig = vit_model(vit_images_orig, labels=None)
            vit_out_flip = vit_model(vit_images_flip, labels=None)
            # 두 출력값의 평균
            vit_tta_cosine = (vit_out_orig + vit_out_flip) / 2.0
            
            # 결과 저장
            all_eff_tta_logits.append(eff_tta_cosine.cpu())
            all_vit_tta_logits.append(vit_tta_cosine.cpu())
            all_labels.append(labels.cpu())

# 텐서 병합
all_eff_tta_logits = torch.cat(all_eff_tta_logits)
all_vit_tta_logits = torch.cat(all_vit_tta_logits)
all_labels = torch.cat(all_labels)

# --- 단일 모델 TTA 성능 확인 ---
_, eff_preds = all_eff_tta_logits.max(1)
eff_acc = eff_preds.eq(all_labels).sum().item() / len(all_labels)

_, vit_preds = all_vit_tta_logits.max(1)
vit_acc = vit_preds.eq(all_labels).sum().item() / len(all_labels)

print("\n=== 단일 모델 성능 (TTA 적용) ===")
print(f"EfficientNet (TTA) : {eff_acc:.4f}")
print(f"ViT (TTA)          : {vit_acc:.4f}")

# 2. TTA 앙상블 가중치 전수 조사
print("\n=== 최종 TTA 앙상블 전수 조사 ===")
best_acc = 0.0
best_weight = 0.0

for w in range(1, 10):
    w_eff = w / 10.0
    w_vit = 1.0 - w_eff
    
    ensemble_logits = (all_eff_tta_logits * w_eff) + (all_vit_tta_logits * w_vit)
    _, ens_preds = ensemble_logits.max(1)
    
    acc = ens_preds.eq(all_labels).sum().item() / len(all_labels)
    print(f"Eff {w_eff:.1f} : ViT {w_vit:.1f} -> Acc: {acc:.4f}")
    
    if acc > best_acc:
        best_acc = acc
        best_weight = w_eff

print(f"\n★ 최고 TTA 앙상블 성능: {best_acc:.4f} (Eff 가중치: {best_weight:.1f})")

[Extracting TTA Logits]:   0%|          | 0/17 [00:00<?, ?it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
[Extracting TTA Logits]:  35%|███▌


=== 단일 모델 성능 (TTA 적용) ===
EfficientNet (TTA) : 0.8650
ViT (TTA)          : 0.8573

=== 최종 TTA 앙상블 전수 조사 ===
Eff 0.1 : ViT 0.9 -> Acc: 0.8582
Eff 0.2 : ViT 0.8 -> Acc: 0.8582
Eff 0.3 : ViT 0.7 -> Acc: 0.8621
Eff 0.4 : ViT 0.6 -> Acc: 0.8650
Eff 0.5 : ViT 0.5 -> Acc: 0.8689
Eff 0.6 : ViT 0.4 -> Acc: 0.8708
Eff 0.7 : ViT 0.3 -> Acc: 0.8717
Eff 0.8 : ViT 0.2 -> Acc: 0.8660
Eff 0.9 : ViT 0.1 -> Acc: 0.8640

★ 최고 TTA 앙상블 성능: 0.8717 (Eff 가중치: 0.7)
